# Does pre-match context improve a simple benchmark?

Use the 19 records from 2025–26 to keep target wording consistent. Freeze three features (weekend, kickoff hour, restricted-ballot eligibility) and ridge alpha=10 before evaluating. Final schedule times may be rescheduled: the benchmark represents near-match planning, not booking-date forecasting.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
d = pd.read_csv(ROOT / 'data/processed/fixtures.csv')
ctx = json.loads((ROOT / 'data/curated/context.json').read_text())


In [2]:
from analysis.modeling import evaluate
model, predictions = evaluate(d)
display(pd.DataFrame(model['scores']))
print(model['validation'])
print(model['decision'])

,model,mae,rmse,n_test
0,expanding_mean,436.51,618.21,9
1,last_fixture,1110.11,1197.71,9
2,ridge_context,454.30,652.31,9


Three expanding windows, 10/13/16 training fixtures, next 3 fixtures each. Fixed alpha=10; no tuning on test data. Nine unique held-out fixtures. Baselines recalculated using training only.
Exploratory benchmark only. Nineteen irregular fixtures in a consistent target scope cannot establish annual seasonality or reliable stationarity. No ARIMA/SARIMA or operational future forecast is fitted.


In [3]:
display(predictions)
assert (predictions.train_end < predictions.test_date).all()

,model,fold,train_end,test_date,fixture_id,actual,prediction
0,expanding_mean,10,2026-01-01,2026-01-17,2025-26-11,1655,1366.6000
1,expanding_mean,10,2026-01-01,2026-01-31,2025-26-12,1367,1366.6000
2,expanding_mean,10,2026-01-01,2026-02-08,2025-26-13,867,1366.6000
3,last_fixture,10,2026-01-01,2026-01-17,2025-26-11,1655,2147.0000
4,last_fixture,10,2026-01-01,2026-01-31,2025-26-12,1367,2147.0000
5,last_fixture,10,2026-01-01,2026-02-08,2025-26-13,867,2147.0000
6,ridge_context,10,2026-01-01,2026-01-17,2025-26-11,1655,1441.0909
7,ridge_context,10,2026-01-01,2026-01-31,2025-26-12,1367,1520.1907
8,ridge_context,10,2026-01-01,2026-02-08,2025-26-13,867,989.2948
9,expanding_mean,13,2026-02-08,2026-02-28,2025-26-14,1408,1350.3846


## Interpretation

The expanding mean has lower held-out MAE than the ridge model on nine unique test fixtures. This tiny evaluation is not evidence of operational readiness. Gaps of 5–29 days and only one comparable season do not support annual seasonal ARIMA. Stationarity is not established; do not force a low-power test or interpolate fictional observations.

Sources and grain boundaries: [DATA_PROVENANCE.md](../DATA_PROVENANCE.md), [data contract](../docs/DATA_CONTRACT.md).